In [ ]:
import cv2
import numpy as np

# Input files
video_path = 'adeka_upal.avi'
skeleton_file = 'yolo_skeleton_coordinates.txt'
output_video_path = 'yolo_video_with_skeleton.avi'

# Read all coordinates into a dict: {frame_id: [points]}
skeleton_data = {}

with open(skeleton_file, 'r') as f:
    for line in f:
        parts = list(map(float, line.strip().split(',')))
        frame_id = int(parts[0])
        points = np.array(parts[1:]).reshape(-1, 2)
        skeleton_data[frame_id] = points

# Open input video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

frame_id = 0

def is_valid(pt):
    return pt is not None and not np.isnan(pt[0]) and not np.isnan(pt[1])

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id in skeleton_data:
        pts = skeleton_data[frame_id].astype(int)

        try:
            # Extract points
            spine_top, spine_bottom = pts[0], pts[1]
            l_hip, l_knee, l_ankle = pts[2], pts[3], pts[4]
            r_hip, r_knee, r_ankle = pts[5], pts[6], pts[7]

            # Draw spine
            cv2.line(frame, spine_top, spine_bottom, (255, 0, 0), 2)     # blue

            # Left leg
            cv2.line(frame, l_hip, l_knee, (0, 255, 255), 2)             # yellow
            cv2.line(frame, l_knee, l_ankle, (255, 255, 0), 2)           # light blue

            # Right leg
            cv2.line(frame, r_hip, r_knee, (0, 255, 255), 2)
            cv2.line(frame, r_knee, r_ankle, (255, 255, 0), 2)

        except Exception as e:
            print(f"Could not draw skeleton on frame {frame_id}: {e}")

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Done! Output video saved as {output_video_path}")

✅ Done! Output video saved as yolo_video_with_skeleton.avi
